In [ ]:

!pip install matplotlib




In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_csv('../artifacts/sentiment_analysis.csv')

In [ ]:
data.head()


In [ ]:
##data preprocessing

In [ ]:
data.shape

In [ ]:
data.duplicated().sum()

In [ ]:
data.isnull().sum()

###text preprossesing

In [ ]:
import re
import string

conver upper case to lower case


In [ ]:
data["tweet"].head()

In [ ]:
data["tweet"] = data["tweet"].apply(lambda x:" ".join(x.lower() for x in x.split()))

In [ ]:
data["tweet"].head()

remove links


In [ ]:
data["tweet"] = data["tweet"].apply(lambda x:" ".join(re.sub(r'^https?:\/\/.*[\r\n]*','',x,flags =re.MULTILINE) for x in x.split()))
data["tweet"].head()

def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation,'')
    return text


data["tweet"] = data["tweet"].apply(remove_punctuations)

data["tweet"] = data["tweet"].str.replace('\d+','',regex = True)



##remove puncthuations

In [ ]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation,'')
    return text


data["tweet"] = data["tweet"].apply(remove_punctuations)

In [ ]:
data["tweet"].tail(10)

##remove numbers

In [ ]:
data["tweet"] = data["tweet"].str.replace('\d+','',regex = True)

In [ ]:
data["tweet"].tail(10)

remove stop words

In [ ]:
! pip install nltk

In [ ]:
import nltk

In [ ]:
nltk.download("stopwords", download_dir ="../static/modle")

In [ ]:
with open ("../static/modle/corpora/stopwords/english",'r') as file:
    sw = file.read().splitlines()


In [ ]:
sw


In [ ]:
data["tweet"] = data["tweet"].apply(lambda x:" ".join(x for x in x.split() if x not in sw))
data["tweet"].head()

stemming(get base word)


In [ ]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [ ]:
data["tweet"] = data["tweet"].apply(lambda x :" ".join(ps.stem(x) for x in x.split()))

In [ ]:
data["tweet"].head()


vocabulary buliding


In [ ]:
from collections import Counter
vocab = Counter()


In [ ]:
for sentence in data['tweet']:
    vocab.update(sentence.split())

             

In [ ]:
vocab


In [ ]:
len(vocab)

In [ ]:
tokens = [key for key in vocab if vocab[key]>10]

In [ ]:
len(tokens)

In [ ]:
def save_vocabulary(lines,filename):
    data = '\n'.join(lines)
    file = open(filename,'w',encoding = "utf-8")
    file.write(data)
    file.close()
save_vocabulary(tokens,'../static/modle/vocabulary.txt')


### divde test trin data set

In [ ]:
data

In [ ]:

!pip install scikit-learn

x = data['tweet']
y = data['label']



In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test = train_test_split(x,y,test_size = 0.2)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

## vectorization

In [ ]:
import numpy as np

def vectorizer(ds, vocabulary):
    # Create a mapping of vocabulary for faster lookup
    vocab_dict = {word: idx for idx, word in enumerate(vocabulary)}
    
    vectorized_lst = []
    for sentence in ds:
        # Initialize a zero vector for the sentence
        sentence_vector = np.zeros(len(vocabulary), dtype=np.float16)
        
        # Split the sentence into words and mark corresponding indices
        for word in sentence.split():
            if word in vocab_dict:
                sentence_vector[vocab_dict[word]] = 1
        
        vectorized_lst.append(sentence_vector)
    
    # Convert the list of vectors to a NumPy array
    vectorized_lst_new = np.array(vectorized_lst, dtype=np.float32)
    
    return vectorized_lst_new


In [ ]:
vectorized_x_train = vectorizer(X_train,tokens)

In [ ]:

vectorized_x_test = vectorizer(X_test,tokens)


In [ ]:
Y_train.value_counts()

hadle implese data

In [ ]:

!pip install imbalanced-learn
from imblearn.over_sampling import SMOTE
smote = SMOTE()
vetorized_xtrain_smote,Y_train_smote =smote.fit_resample(vectorized_x_train,Y_train)
print(vetorized_xtrain_smote.shape,Y_train_smote.shape)

In [ ]:
smoteY_train_.value_counts()

## modle traning and evalution

In [ ]:


from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier


In [ ]:
from sklearn.metrics import accuracy_score,f1_score,precision_score,recall_score

def training_scores(y_act, y_pred):
    acc = round(accuracy_score(y_act, y_pred),3)
    pr = round(precision_score(y_act, y_pred),3)
    rec =round(recall_score(y_act, y_pred),3)
    f1 =round(f1_score(y_act, y_pred),3)
    print(f'Traning scores:\n\tAccuracy ={acc}\n\tPrecision ={pr}\n\tRecall = {rec}\n\tF1-score = {f1}')

def validation_scores(y_act, y_pred):
    acc = round(accuracy_score(y_act, y_pred),3)
    pr = round(precision_score(y_act, y_pred),3)
    rec =round(recall_score(y_act, y_pred),3)
    f1 =round(f1_score(y_act, y_pred),3)
    print(f'Testing scores:\n\tAccuracy ={acc}\n\tPrecision ={pr}\n\tRecall = {rec}\n\tF1-score = {f1}')
    
    

logiistric regrassion

In [ ]:

from sklearn.linear_model import LogisticRegression
# Train the model
lr =  LogisticRegression()
lr.fit(vetorized_xtrain_smote, Y_train_smote)

# Generate predictions
y_train_pred = lr.predict(vetorized_xtrain_smote)
y_test_predit = lr.predict(vectorized_x_test)


# Evaluate the predictions
training_scores(Y_train_smote, y_train_pred)
validation_scores(Y_test,y_test_predit)


naive base

In [ ]:
from sklearn.naive_bayes import MultinomialNB

# Train the model
nlb = MultinomialNB()
nlb.fit(vetorized_xtrain_smote, Y_train_smote)

# Generate predictions
y_train_pred = nlb.predict(vetorized_xtrain_smote)
y_test_predit = nlb.predict(vectorized_x_test)


# Evaluate the predictions
training_scores(Y_train_smote, y_train_pred)
validation_scores(Y_test,y_test_predit)


decistion tree

In [ ]:
tr = DecisionTreeClassifier()
tr.fit(vetorized_xtrain_smote, Y_train_smote)

# Generate predictions
y_train_pred = tr.predict(vetorized_xtrain_smote)
y_test_predit = tr.predict(vectorized_x_test)


# Evaluate the predictions
training_scores(Y_train_smote, y_train_pred)
validation_scores(Y_test,y_test_predit)

RandomForest

In [ ]:
rf =RandomForestClassifier()
rf.fit(vetorized_xtrain_smote, Y_train_smote)

# Generate predictions
y_train_pred = rf.predict(vetorized_xtrain_smote)
y_test_predit = rf.predict(vectorized_x_test)


# Evaluate the predictions
training_scores(Y_train_smote, y_train_pred)
validation_scores(Y_test,y_test_predit)


support vector

In [ ]:
svm =SVC()
svm.fit(vetorized_xtrain_smote, Y_train_smote)

# Generate predictions
y_train_pred = svm.predict(vetorized_xtrain_smote)
y_test_predit = svm.predict(vectorized_x_test)


# Evaluate the predictions
training_scores(Y_train_smote, y_train_pred)
validation_scores(Y_test,y_test_predit)

In [ ]:
import pickle
with open('../static/modle/modle.pickle','wb') as file:
    pickle.dump(svm,file)